In [1]:
import csv
import re
from pathlib import Path

COURSE_FILE = "../CsvForDB/Course.csv"
KEYWORDS_FILE = "../CsvForDB/Keywords.csv"
OUTPUT_FILE = "../CsvForDB/CourseKeywords.csv"


def tokenize(text: str) -> set[str]:
    """
    Convert text to a set of lowercase word tokens.
    Example: 'AI Robotics' -> {'ai', 'robotics'}
    """
    return set(re.findall(r"\b\w+\b", text.lower()))


def main() -> None:
    course_path = Path(COURSE_FILE)
    keywords_path = Path(KEYWORDS_FILE)

    if not course_path.exists():
        raise FileNotFoundError(f"Missing file: {COURSE_FILE}")
    if not keywords_path.exists():
        raise FileNotFoundError(f"Missing file: {KEYWORDS_FILE}")

    # Load keywords
    keywords = []
    with keywords_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            keyword_id = row["keywordId"].strip()
            term = row["normalized_term"].strip().lower()
            keywords.append((keyword_id, term))

    # Find matches
    matches = []
    with course_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            course_id = row["id"].strip()
            title = row["title"].strip()
            title_tokens = tokenize(title)

            for keyword_id, term in keywords:
                if term in title_tokens:
                    matches.append({
                        "courseId": course_id,
                        "keywordId": keyword_id
                    })

    # Write output
    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["courseId", "keywordId"])
        writer.writeheader()
        writer.writerows(matches)

    print(f"Wrote {len(matches)} matches to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Wrote 4955 matches to ../CsvForDB/CourseKeywords.csv
